In [ ]:
# import required libraries
import pandas as pd

In [ ]:
# read the data
data_df = pd.read_csv('250807_FinalCleanedData_PT.csv')

# filter the data
print(data_df.keys())
data_df.head()

Index(['Unnamed: 0.1', 'caseid', 'weight', 'wave', 'state', 'district',
       'urban', 'gender', 'age', 'caste',
       ...
       'A63_p2', 'A63_p90', 'A63_p95', 'A63_p98', 'A63_sum', 'A63_mean',
       'A63_min', 'A63_max', 'A63_count', 'split'],
      dtype='object', length=608)


,Unnamed: 0.1,caseid,weight,wave,state,district,urban,gender,age,caste,...,A63_p2,A63_p90,A63_p95,A63_p98,A63_sum,A63_mean,A63_min,A63_max,A63_count,split
0,0,10001,0.840547,2024,NE,East Khasi Hills,Urban,Male,18-29,Scheduled Castes/Tribes,...,-0.124567,0.103406,0.124567,0.147697,43582.604336,0.012406,-0.267958,0.327812,3519450,train
1,1,10002,1.360886,2024,UT,Leh,Rural,Female,45+,Scheduled Castes/Tribes,...,-0.206936,0.093564,0.141730,0.267958,-117536.992772,-0.024344,-0.318893,0.346021,4837249,train
2,2,10002,1.360886,2024,UT,Leh,Rural,Female,45+,Scheduled Castes/Tribes,...,-0.221453,0.103406,0.130165,0.160000,-248714.921995,-0.038288,-0.327812,0.318893,6508506,train
3,3,10003,0.640417,2024,UT,Daman & Diu,Rural,Male,30-44,Other Castes,...,0.032541,0.221453,0.236463,0.244152,10633.937208,0.124858,-0.051734,0.355309,86148,train
4,4,10003,0.640417,2024,UT,Daman & Diu,Rural,Male,30-44,Other Castes,...,-0.022207,0.206936,0.244152,0.259900,4060.501217,0.087937,-0.113741,0.355309,47012,train


## Define column names for different models

In [ ]:
# create keys for all four models
features_list_model1 = ['age', 'gender', 'urban', 'caste']
features_list_model2 = features_list_model1 + ['mean_monthly_avg_rd_MD', 'precip_sum_mm', 'precip_mean_mm', 'mean_tmax', 'district_flood_sentiment', 'state_flood_sentiment']
features_list_model3 = features_list_model1 + [
    f'{var}_{metric}'
    for var in sorted(set([
        data.split('_')[0]
        for data in data_df.loc[:, data_df.columns.str.startswith('A')].keys()
    ]))
    for metric in ['min', 'p2', 'mean', 'p98', 'max', 'sum', 'count']
]
features_list_model4 = list(set(features_list_model2 + features_list_model3))

flood_y_key = 'n7fy23_recode'
drought_y_key = 'n7dy23_recode'

## Prepare data for the model

In [ ]:
%%time

%%time
from sklearn.preprocessing import LabelEncoder

# Encode categorical variables FIRST (on full dataset)
data_encoded = data_df.copy()
encoders = {}

for var in ['age', 'gender', 'urban', 'caste']:
    encoders[var] = LabelEncoder()
    data_encoded[var] = encoders[var].fit_transform(data_df[var])

# Split the encoded data (general variables)
X_train_encoded = data_encoded[data_encoded['split'] == 'train']
X_test_encoded = data_encoded[data_encoded['split'] == 'test']
y_train = data_encoded[data_encoded['split'] == 'train'][flood_y_key]
y_test = data_encoded[data_encoded['split'] == 'test'][flood_y_key]

print(f"X_train_encoded shape: {X_train_encoded.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test_encoded shape: {X_test_encoded.shape}")
print(f"y_test shape: {y_test.shape}")

X_train_encoded shape: (9376, 608)
y_train shape: (9376,)
X_test_encoded shape: (2047, 608)
y_test shape: (2047,)
CPU times: user 6.36 s, sys: 95.8 ms, total: 6.45 s
Wall time: 802 ms
CPU times: user 6.36 s, sys: 95.9 ms, total: 6.45 s
Wall time: 804 ms


## Train model 1 - demographic variables only

In [ ]:
%%time

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
import pandas as pd
import time

# Extract Model 1 specific features
X_train_encoded_model1 = X_train_encoded[features_list_model1]
X_test_encoded_model1 = X_test_encoded[features_list_model1]

models = {
   'RandomForest': (RandomForestClassifier(random_state=42), {
       'n_estimators': [10, 50, 100, 200], 'max_depth': [10, 20, None]}),

   'GradientBoosting': (GradientBoostingClassifier(random_state=42), {
       'n_estimators': [100, 200], 'learning_rate': [0.1, 0.2], 'max_depth': [3, 5]}),

   'LogisticRegression': (LogisticRegression(random_state=42, max_iter=1000), {
       'C': [0.1, 1, 10], 'penalty': ['l1', 'l2'], 'solver': ['liblinear']})
}

results = []
for name, (model, params) in models.items():
   print(f"Training {name}...")
   start = time.time()

   grid = GridSearchCV(model, params, cv=5, scoring='accuracy')
   grid.fit(X_train_encoded_model1, y_train)

   train_pred = grid.predict(X_train_encoded_model1)
   test_pred = grid.predict(X_test_encoded_model1)
   train_proba = grid.predict_proba(X_train_encoded_model1)[:, 1]
   test_proba = grid.predict_proba(X_test_encoded_model1)[:, 1]

   results.append({
       'Model 1': name,
       'Train_Accuracy': accuracy_score(y_train, train_pred),
       'Test_Accuracy': accuracy_score(y_test, test_pred),
       'Train_Precision': precision_score(y_train, train_pred),
       'Test_Precision': precision_score(y_test, test_pred),
       'Train_Recall': recall_score(y_train, train_pred),
       'Test_Recall': recall_score(y_test, test_pred),
       'Train_F1': f1_score(y_train, train_pred),
       'Test_F1': f1_score(y_test, test_pred),
       'Train_AUC': roc_auc_score(y_train, train_proba),
       'Test_AUC': roc_auc_score(y_test, test_proba)
   })

   print(f"{name} - Test Acc: {results[-1]['Test_Accuracy']:.3f}, Test AUC: {results[-1]['Test_AUC']:.3f}")
   print(f"Time: {time.time()-start:.1f}s\n")

# Create DataFrame
model1_metrics = pd.DataFrame(results)

print("FINAL RESULTS:")
model1_metrics.round(3).set_index('Model 1').T

Training RandomForest...
RandomForest - Test Acc: 0.523, Test AUC: 0.548
Time: 8.4s

Training GradientBoosting...
GradientBoosting - Test Acc: 0.513, Test AUC: 0.546
Time: 13.8s

Training LogisticRegression...
LogisticRegression - Test Acc: 0.541, Test AUC: 0.556
Time: 0.3s

FINAL RESULTS:
CPU times: user 22.7 s, sys: 1.72 s, total: 24.5 s
Wall time: 22.7 s


Model 1,RandomForest,GradientBoosting,LogisticRegression
Train_Accuracy,0.534,0.534,0.533
Test_Accuracy,0.523,0.513,0.541
Train_Precision,0.515,0.516,0.513
Test_Precision,0.525,0.510,0.556
Train_Recall,0.324,0.292,0.308
Test_Recall,0.339,0.298,0.339
Train_F1,0.398,0.373,0.385
Test_F1,0.412,0.376,0.421
Train_AUC,0.541,0.541,0.534
Test_AUC,0.548,0.546,0.556


## Train model 2 - demographic variables + physical variables

In [ ]:
%%time

from sklearn.ensemble import GradientBoostingClassifier, ExtraTreesClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
import pandas as pd
import time

# Extract Model 1 specific features
X_train_encoded_model2 = X_train_encoded[features_list_model2]
X_test_encoded_model2 = X_test_encoded[features_list_model2]

results = []
for name, (model, params) in models.items():
   print(f"Training {name}...")
   start = time.time()

   grid = GridSearchCV(model, params, cv=5, scoring='accuracy')
   grid.fit(X_train_encoded_model2, y_train)

   train_pred = grid.predict(X_train_encoded_model2)
   test_pred = grid.predict(X_test_encoded_model2)
   train_proba = grid.predict_proba(X_train_encoded_model2)[:, 1]
   test_proba = grid.predict_proba(X_test_encoded_model2)[:, 1]

   results.append({
       'Model 2': name,
       'Train_Accuracy': accuracy_score(y_train, train_pred),
       'Test_Accuracy': accuracy_score(y_test, test_pred),
       'Train_Precision': precision_score(y_train, train_pred),
       'Test_Precision': precision_score(y_test, test_pred),
       'Train_Recall': recall_score(y_train, train_pred),
       'Test_Recall': recall_score(y_test, test_pred),
       'Train_F1': f1_score(y_train, train_pred),
       'Test_F1': f1_score(y_test, test_pred),
       'Train_AUC': roc_auc_score(y_train, train_proba),
       'Test_AUC': roc_auc_score(y_test, test_proba)
   })

   print(f"{name} - Test Acc: {results[-1]['Test_Accuracy']:.3f}, Test AUC: {results[-1]['Test_AUC']:.3f}")
   print(f"Time: {time.time()-start:.1f}s\n")

# Create DataFrame
model2_metrics = pd.DataFrame(results)

print("FINAL RESULTS:")
model2_metrics.round(3).set_index('Model 2').T

Training RandomForest...
RandomForest - Test Acc: 0.535, Test AUC: 0.566
Time: 30.0s

Training GradientBoosting...
GradientBoosting - Test Acc: 0.560, Test AUC: 0.571
Time: 45.7s

Training LogisticRegression...
LogisticRegression - Test Acc: 0.537, Test AUC: 0.576
Time: 2.3s

FINAL RESULTS:
CPU times: user 1min 26s, sys: 57.9 s, total: 2min 24s
Wall time: 1min 18s


Model 2,RandomForest,GradientBoosting,LogisticRegression
Train_Accuracy,0.711,0.623,0.539
Test_Accuracy,0.535,0.560,0.537
Train_Precision,0.712,0.613,0.528
Test_Precision,0.536,0.561,0.573
Train_Recall,0.656,0.558,0.277
Test_Recall,0.424,0.493,0.236
Train_F1,0.682,0.584,0.364
Test_F1,0.474,0.525,0.334
Train_AUC,0.785,0.677,0.544
Test_AUC,0.566,0.571,0.576


## Train model 3 - demographic variables + satellite embeddings

In [ ]:
%%time

from sklearn.ensemble import GradientBoostingClassifier, ExtraTreesClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
import pandas as pd
import time

# Extract Model 1 specific features
X_train_encoded_model3 = X_train_encoded[features_list_model3]
X_test_encoded_model3 = X_test_encoded[features_list_model3]

results = []
for name, (model, params) in models.items():
   print(f"Training {name}...")
   start = time.time()

   grid = GridSearchCV(model, params, cv=5, scoring='accuracy')
   grid.fit(X_train_encoded_model3, y_train)

   train_pred = grid.predict(X_train_encoded_model3)
   test_pred = grid.predict(X_test_encoded_model3)
   train_proba = grid.predict_proba(X_train_encoded_model3)[:, 1]
   test_proba = grid.predict_proba(X_test_encoded_model3)[:, 1]

   results.append({
       'Model 3': name,
       'Train_Accuracy': accuracy_score(y_train, train_pred),
       'Test_Accuracy': accuracy_score(y_test, test_pred),
       'Train_Precision': precision_score(y_train, train_pred),
       'Test_Precision': precision_score(y_test, test_pred),
       'Train_Recall': recall_score(y_train, train_pred),
       'Test_Recall': recall_score(y_test, test_pred),
       'Train_F1': f1_score(y_train, train_pred),
       'Test_F1': f1_score(y_test, test_pred),
       'Train_AUC': roc_auc_score(y_train, train_proba),
       'Test_AUC': roc_auc_score(y_test, test_proba)
   })

   print(f"{name} - Test Acc: {results[-1]['Test_Accuracy']:.3f}, Test AUC: {results[-1]['Test_AUC']:.3f}")
   print(f"Time: {time.time()-start:.1f}s\n")

# Create DataFrame
model3_metrics = pd.DataFrame(results)

print("FINAL RESULTS:")
model3_metrics.round(3).set_index('Model 3').T

Training RandomForest...
